In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from tqdm import tqdm
import os
import yaml
import zuko
from helpers.data_transforms import preprocess_data, inverse_preprocess_data, load_in_data

from helpers.models.DNN import count_parameters
from helpers.evaluation import run_eval_suite_BDTs, run_eval_suite_R
from helpers.plotting import plot_hists_1d, plot_corner_hist_2d, make_2d_plots
from helpers.material_map import apply_material_map_hybrid, build_masked_datasets
from helpers.flow import build_z_lookup, snap_z_to_detector

plt.style.use("../science.mplstyle")

In [2]:
BIN_BOUND = 5
NUM_BINS = 200
NUM_FEATURES = 5
num_BDTs = 1

collections = [
    "InnerTrackerBarrelCollection",
    "OuterTrackerBarrelCollection",     
   "VertexBarrelCollection",   
]


feature_order_barrel =[0,4,1,2,3,6,7]
feature_indices_dict_barrel = {
   "r":2,
    "phi": 3,
    "z":4,
    "side":5,
    "layer":6,
}

feature_order_dict = {
     "InnerTrackerBarrelCollection":feature_order_barrel,
    "OuterTrackerBarrelCollection":feature_order_barrel,
    "VertexBarrelCollection":feature_order_barrel,

}


feature_indices_dict = {
     "InnerTrackerBarrelCollection":feature_indices_dict_barrel,
    "OuterTrackerBarrelCollection":feature_indices_dict_barrel,
    "VertexBarrelCollection":feature_indices_dict_barrel,

}



ZUKO_ID = "NCSF"
NAME = "cond2"
NUM_COND_INPUTS = 2
SEED = 8



FEATURES = "rphi"
working_dir = "/pscratch/sd/r/rmastand/muon_collider"
log_vars = []



In [3]:
# load in samples

all_data_dir, all_samples_dir = {}, {}
bins_dict, bins_dict_preproc = {}, {}
aucs = {}
feature_labels_dict = {}

for col_name in collections:

    small_id = ''.join([c for c in col_name if c.isupper()])
    X, feature_labels = load_in_data([col_name], FEATURES, working_dir, 1, NUM_COND_INPUTS, feature_order_dict[col_name])
    
    all_data_dir[col_name] = X
    all_samples_dir[col_name] = np.load(f"{working_dir}/zuko_outputs/{ZUKO_ID}/{small_id}_{NAME}/flow_samples.npy")
    with open(f"{working_dir}/zuko_outputs/{ZUKO_ID}/{small_id}_{NAME}/results.txt") as ifile:
       aucs[col_name]=  ifile.readlines()[-1]
    feature_labels_dict[col_name] = feature_labels


    

    bins_dict[col_name] = {}
    bins_dict_preproc[col_name] = {i:np.linspace(-BIN_BOUND, BIN_BOUND, NUM_BINS) for i in range(all_data_dir[col_name].shape[1])}
    
    for i in range(all_data_dir[col_name].shape[1]):
        if i in log_vars:
            bins_dict[col_name][i] = np.logspace(np.log10(0.9*np.min(all_data_dir[col_name][:,i])), np.log10(1.1*np.max(all_data_dir[col_name][:,i])), NUM_BINS) 
        else:
            bins_dict[col_name][i] = np.linspace(np.min(all_data_dir[col_name][:,i] - 3), np.max(all_data_dir[col_name][:,i] + 3), NUM_BINS) 

In [ ]:
from helpers.material_map import torch_barrel_material_penalty
import torch

for col_name in collections:

    real_mmap = torch_barrel_material_penalty(
            torch.tensor(all_data_dir[col_name]),
            col_name,
            feature_indices_dict,
            softness=1.0,
        
        )

    samples_mmap = torch_barrel_material_penalty(
            torch.tensor(all_samples_dir[col_name]),
            col_name,
            feature_indices_dict,
            softness=1.0,
        )
    print(col_name, real_mmap, samples_mmap)

InnerTrackerBarrelCollection tensor(0.6847, dtype=torch.float64) tensor(nan)


In [ ]:
bins = 1000
for col_name in collections:

    print(col_name)

    plt.figure(figsize = (10, 10))

    plt.hist2d(all_data_dir[col_name][:,feature_indices_dict[col_name]["r"]]*np.cos(all_data_dir[col_name][:,feature_indices_dict[col_name]["phi"]]), 
               all_data_dir[col_name][:,feature_indices_dict[col_name]["r"]]*np.sin(all_data_dir[col_name][:,feature_indices_dict[col_name]["phi"]]), 
               bins=(bins,bins),
              norm="log",)
    plt.show()
   